In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_huggingface import HuggingFaceEmbeddings
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'


In [ ]:
import bs4

# Hub（新版：移到 langchain-classic）
from langchain_classic import hub

# Splitters（新版：langchain_text_splitters）
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Community integrations
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

# Core building blocks
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# OpenAI integration
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
import os
from langchain_deepseek import ChatDeepSeek

# 确保环境变量存在，或直接在下面传 api_key 参数（如果该版本支持）
#assert os.getenv("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

llm = ChatDeepSeek(
    model="deepseek-chat",   # DeepSeek-V3: 支持 tools/structured output
    temperature=0,
    api_key="sk-3d4dbf63d7704840976b05ddbb9957a7"
)

In [ ]:
from langchain_community.vectorstores import FAISS
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed
vectorstore=FAISS.from_documents(documents=splits , embedding=embeddings)
retriever=vectorstore.as_retriever(search_kwargs={"k":4})

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = hub.pull("rlm/rag-prompt")


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
rag_chain.invoke("What is Task Decomposition?")

In [ ]:
def format_docs_null(docs):
    return ""

In [ ]:
# 测试retriever是否生效
rag_chain = (
    {"context": retriever| format_docs_null, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke("What is Task Decomposition?")

In [ ]:
q = "What is Task Decomposition?"
docs = retriever.invoke(q) 
print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:500])